# Experiment 2 revised pilot — population launch

Pinned to `d60a1996a634b21a5e6d9fd61ef2070d59ec728f`. Runs the owner-approved 20-run population with one reference run per T4 (two workers). Each worker runs to a 10-hour session budget; incomplete runs resume bit-exactly from `/kaggle/working/exp2_pilot` when the notebook is re-run with the previous output mounted as the `exp2-pilot-state` dataset. Never assumes the full population fits one Kaggle session or the weekly GPU quota.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ.update({"JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl", "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled", "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"})
import signal
signal.signal(signal.SIGALRM, lambda signum, frame: (_ for _ in ()).throw(TimeoutError("Experiment 2 revised pilot qualification exceeded its limit")))
signal.alarm(4 * 60 * 60)
import platform
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path

SOURCE_SHA = "d60a1996a634b21a5e6d9fd61ef2070d59ec728f"
SOURCE_BRANCH = "codex/exp2-revised-pilot-20261112"
REPO = Path("/kaggle/temp/scaling-drl-research")
REPO.parent.mkdir(parents=True, exist_ok=True)

def wait_for_dns(host, attempts=6, delay=10):
    for attempt in range(attempts):
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            print("DNS available:", host)
            return
        except socket.gaierror as exc:
            print(f"DNS attempt {attempt + 1}/{attempts} failed for {host}: {exc}", flush=True)
            if attempt + 1 < attempts:
                time.sleep(delay)
    raise RuntimeError(f"Kaggle cannot resolve {host}. Enable Internet and retry.")

for host in ("github.com", "pypi.org", "files.pythonhosted.org"):
    wait_for_dns(host)

if not REPO.exists():
    for attempt in range(3):
        with tempfile.TemporaryDirectory(prefix="exp2-clone-", dir=REPO.parent) as tmp:
            destination = Path(tmp) / "source"
            result = subprocess.run(["git", "clone", "--quiet", "--single-branch", "--branch", SOURCE_BRANCH, "https://github.com/kaves29/scaling-drl-research.git", str(destination)], text=True, capture_output=True)
            if result.returncode == 0:
                shutil.move(str(destination), str(REPO))
                break
            print(f"Git clone attempt {attempt + 1}/3 failed: {result.stderr.strip()}", flush=True)
        if attempt < 2:
            time.sleep(10)
    else:
        raise RuntimeError("Could not clone the pinned source after three attempts.")
subprocess.run(["git", "fetch", "--quiet", "origin", SOURCE_SHA], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_SHA

requirements = (REPO / "main/requirements.txt").read_text()
install_dm_control_without_optional_mazes = sys.version_info >= (3, 13)
if install_dm_control_without_optional_mazes:
    replacements = {"numpy==1.26.4": "numpy==2.1.3", "pandas==2.1.4": "pandas==2.2.3", "matplotlib==3.9.0": "matplotlib==3.9.2", "Shimmy==2.0.0": "Shimmy==1.3.0"}
    for old, new in replacements.items():
        assert requirements.count(old) == 1, f"Expected exactly one {old} pin"
        requirements = requirements.replace(old, new)
    for unused_pin in ("labmaze==1.0.6", "dm_control==1.0.38"):
        assert requirements.count(unused_pin) == 1, f"Expected exactly one {unused_pin} pin"
        requirements = requirements.replace(unused_pin + "\n", "")
runtime_requirements = Path("/kaggle/temp/exp2-runtime-requirements.txt")
runtime_requirements.write_text(requirements)
print("Python runtime:", platform.python_version(), "; installing:", runtime_requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(runtime_requirements)], check=True)
if install_dm_control_without_optional_mazes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "dm_control==1.0.38"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=True, stdout=subprocess.DEVNULL)
    print("Removed Kaggle CUDA 13 JAX plugins to avoid conflict with the pinned CUDA 12 backend.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]==0.4.34"], check=True)

import importlib.metadata as package_metadata
import flax
import gymnasium
import jax
import matplotlib
import mujoco
import numpy
if not hasattr(numpy, "issctype"):
    def _numpy_issctype(rep):
        return isinstance(rep, type) and issubclass(rep, numpy.generic)
    numpy.issctype = _numpy_issctype
import optax
import pandas
from dm_control import suite as dmc_suite
assert ("dog", "run") in dmc_suite.ALL_TASKS
print("Core runtime:", {name: package_metadata.version(name) for name in ("jax", "numpy", "flax", "optax", "dm-control", "gymnasium", "mujoco")})
print("Pinned revised-pilot source installed:", SOURCE_SHA)


In [ ]:
# Set rendering and JAX selection before importing either library.
os.environ.update({
    "JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl",
    "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled",
    "XDG_CACHE_HOME": "/kaggle/temp/.cache", "MPLCONFIGDIR": "/kaggle/temp/.mpl",
    "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "PYTHONDONTWRITEBYTECODE": "1",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
})
import jax
import numpy as np
names = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).splitlines()
assert len(names) == 2 and all("T4" in name for name in names), f"Expected 2 T4 GPUs, got {names}"
devices = jax.devices("gpu")
assert jax.default_backend() == "gpu" and len(devices) == 2, devices
for device in devices:
    with jax.default_device(device):
        value = jax.device_put(np.ones((128, 128), dtype=np.float32), device)
        out = jax.jit(lambda array: array @ array)(value)
        out.block_until_ready()
        assert out.devices() == {device}
print("GPU inventory:", names)
print("Verified JAX dispatch on both T4 devices:", devices)

import threading
class _Heartbeat:
    """Keep nbclient's 4-second IOPub timer alive during quiet long cells."""
    def __init__(self, label, interval=2):
        self.label, self.interval, self.count = label, interval, 0
        self.done = threading.Event()
        self.thread = threading.Thread(target=self._beat, daemon=True)
        self.thread.start()
    def _beat(self):
        while not self.done.wait(self.interval):
            self.count += 1
            print(f"[heartbeat] {self.label} {self.count}", flush=True)
    def stop(self):
        self.done.set()
        self.thread.join(timeout=1)

def heartbeat(label):
    return _Heartbeat(label)


In [ ]:
# Restore prior session state if the output is mounted as a dataset.
RESUME = Path("/kaggle/input/exp2-pilot-state")
ROOT = Path("/kaggle/working/exp2_pilot")
if RESUME.exists():
    source = RESUME / "exp2_pilot" if (RESUME / "exp2_pilot").exists() else RESUME
    shutil.copytree(source, ROOT, dirs_exist_ok=True)
    print("Restored prior pilot state from", source)
else:
    print("No prior state dataset; fresh start")
ROOT.mkdir(parents=True, exist_ok=True)
assert (REPO / "main/experiments/exp12/revised_exp2_pilot.py").exists()
print("Output root:", ROOT)


In [ ]:
import time
beat = heartbeat("population-launch")

SESSION_BUDGET = 10 * 60 * 60
started = time.time()
workers = []
for gpu, indices in (("0", range(0, 20, 2)), ("1", range(1, 20, 2))):
    assignment = ",".join(str(i) for i in indices)
    log = Path(f"/kaggle/working/worker_gpu{gpu}.log")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, JAX_PLATFORMS="cuda,cpu",
               WANDB_MODE="disabled", XLA_PYTHON_CLIENT_PREALLOCATE="false")
    code = f"""
import signal, subprocess, sys
from pathlib import Path
indices = [{assignment}]
root = {str(ROOT)!r}
for index in indices:
    remaining = {SESSION_BUDGET} - int(__import__('time').time() - {started!r})
    if remaining <= 600:
        print('SESSION-BUDGET-REACHED before run', index, flush=True)
        break
    print('WORKER-START run', index, 'budget', remaining, flush=True)
    result = subprocess.run([sys.executable, {str(REPO / 'main/scripts/run_revised_exp2_pilot.py')!r},
                             '--run-index', str(index), '--root', root,
                             '--time-budget-seconds', str(max(600, remaining - 300))])
    if result.returncode != 0:
        print('WORKER-RUN-INCOMPLETE run', index, 'code', result.returncode, flush=True)
        continue
    print('WORKER-RUN-COMPLETE run', index, flush=True)
print('WORKER-EXIT', flush=True)
"""
    workers.append(subprocess.Popen([sys.executable, "-c", code], env=env, text=True,
                                    stdout=open(log, "w"), stderr=subprocess.STDOUT))
for worker in workers:
    worker.wait()
print("Worker logs:", [f"/kaggle/working/worker_gpu{g}.log" for g in ("0", "1")])
for path in (Path("/kaggle/working/worker_gpu0.log"), Path("/kaggle/working/worker_gpu1.log")):
    text = path.read_text()
    print(f"=== {path.name} ===")
    print("\n".join(text.splitlines()[-40:]))


In [ ]:
import json
complete = sorted(str(p.parent.relative_to(ROOT)) for p in ROOT.glob("*/analysis.json"))
manifest = {"source_sha": SOURCE_SHA, "root": str(ROOT), "complete_runs": complete,
            "complete_count": len(complete), "expected_count": 20}
Path("/kaggle/working/exp2_pilot_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(json.dumps(manifest, indent=2))
beat.stop()
